# Class 5: ReAct Agents Mini‑Demo (OpenAI / LangChain)

**Goal:** Learn to use LangChain's ReAct agent to answer questions by reasoning step-by-step and interacting with tools.

**What you'll learn**
 - How the ReAct agent works (reasoning+acting)
 - Defining and using tools with a ReAct agent
 - Observing the agent's thought process and tool usage

*Tip:* Make sure your `OPENAI_API_KEY` is set in your environment or notebook secrets!

In [16]:
import os
import requests
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

In [2]:
assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY in environment."
CONVEX_BASE = "https://standing-fish-574.convex.site"

In [3]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [4]:
class CityQuery(BaseModel):
    q: str = Field(..., description="Case-insensitive substring of the city name, e.g., 'Tokyo'.")

@tool("city_lookup", args_schema=CityQuery)
def city_lookup(q: str) -> dict:
    """Return up to 5 city matches with IATA codes from the Convex dataset."""
    resp = requests.get(f"{CONVEX_BASE}/reference/cities", timeout=20)
    resp.raise_for_status()
    data = resp.json().get("cities", [])
    ql = q.lower()
    matches = [
        {"city": c["name"], "country": c["country"], "airport_iata": c["airportCode"]}
        for c in data if ql in c["name"].lower()
    ][:5]
    return {"matches": matches}


In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer city/IATA questions by calling tools when needed."),
    ("human", "{input}")
])

In [6]:
agent = create_agent(model, [city_lookup])

In [7]:
messages = prompt.invoke("What is the IATA code for Tokyo's main international airport?").messages

In [8]:
messages

[SystemMessage(content='You answer city/IATA questions by calling tools when needed.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="What is the IATA code for Tokyo's main international airport?", additional_kwargs={}, response_metadata={})]

In [21]:
updated_messages = agent.invoke({"messages": messages})

In [23]:
agent_state = updated_messages["messages"]

agent_state

[SystemMessage(content='You answer city/IATA questions by calling tools when needed.', additional_kwargs={}, response_metadata={}, id='05f0241b-48be-43db-b06b-cba489832236'),
 HumanMessage(content="What is the IATA code for Tokyo's main international airport?", additional_kwargs={}, response_metadata={}, id='3204b036-9974-429f-bc31-30700b72bdf6'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 95, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CS4MnCSiGq7yqajcnm3buuHQe8HHn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--bd1cff17-f749-4d29-a616-3ad069e80c4f-0

In [11]:
for msg in agent.stream({"messages": messages}):
    print(msg)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 95, 'total_tokens': 109, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CS4F7IBRbEUQvFqirmGGwO9wFRZR7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--1e259137-55a2-4e27-96a6-e20facaf81b2-0', tool_calls=[{'name': 'city_lookup', 'args': {'q': 'Tokyo'}, 'id': 'call_EHb5aMf4JmxkZnufFXHJv1Ls', 'type': 'tool_call'}], usage_metadata={'input_tokens': 95, 'output_tokens': 14, 'total_tokens': 109, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}}
{'tools': {'

In [12]:
agent.invoke({"messages": "what did I say earlier"})

{'messages': [HumanMessage(content='what did I say earlier', additional_kwargs={}, response_metadata={}, id='425e360b-326e-4f3c-944d-b6bd1dcd597e'),
  AIMessage(content="I'm unable to recall previous interactions or conversations. However, I can help you with any questions or topics you'd like to discuss now!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 75, 'total_tokens': 102, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CS4GaBmPAJTFe7Gcv2c3muNwZQFGG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--9db68448-fe77-4327-83a0-16fe780c5ae2-0', usage_metadata={'input_tokens': 75, 'output_tokens': 27, 'total

In [28]:
agent_state

[SystemMessage(content='You answer city/IATA questions by calling tools when needed.', additional_kwargs={}, response_metadata={}, id='05f0241b-48be-43db-b06b-cba489832236'),
 HumanMessage(content="What is the IATA code for Tokyo's main international airport?", additional_kwargs={}, response_metadata={}, id='3204b036-9974-429f-bc31-30700b72bdf6'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 95, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CS4MnCSiGq7yqajcnm3buuHQe8HHn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--bd1cff17-f749-4d29-a616-3ad069e80c4f-0

In [29]:
agent.invoke({"messages": agent_state})

{'messages': [SystemMessage(content='You answer city/IATA questions by calling tools when needed.', additional_kwargs={}, response_metadata={}, id='05f0241b-48be-43db-b06b-cba489832236'),
  HumanMessage(content="What is the IATA code for Tokyo's main international airport?", additional_kwargs={}, response_metadata={}, id='3204b036-9974-429f-bc31-30700b72bdf6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 95, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CS4MnCSiGq7yqajcnm3buuHQe8HHn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--bd1cff17-f749-4d29-a616